# 14 · 2-D Dynamic Programming

> Lo stato ha due indici: una cella di griglia `(r, c)`, due stringhe (`i` caratteri della prima, `j` della seconda), indice + capacità, indice + "stato" (posso comprare?), o un intervallo `(l, r)`.
> Prerequisito: [13 · 1-D DP](13%20-%201-D%20Dynamic%20Programming.ipynb) · Transizioni per problema: [Pattern algoritmici › DP 2D](../Fundamentals/Pattern%20algoritmici.md#dp-2d)

## Problemi → cosa ti serve

| Problema | Concetti | Sintassi |
|---|---|---|
| Unique Paths | `dp[r][c]` = somma da sopra e da sinistra | tabella 2D o una sola riga |
| Longest Common Subsequence | due stringhe, caratteri uguali o no | tabella `(m + 1) × (n + 1)`, `s1[i - 1]` |
| Best Time to Buy and Sell Stock with Cooldown | macchina a stati: indice + "ho un'azione?" | `@cache` su `(i, holding)` |
| Coin Change II | conta combinazioni: monete fuori, importi dentro | `dp[x] += dp[x - c]` |
| Target Sum | indice + somma corrente (può essere negativa) | `@cache` o `dict` somma → conteggio |
| Interleaving String | `dp[i][j]` con `s3[i + j - 1]` | `len(s1) + len(s2) != len(s3)` |
| Longest Increasing Path in a Matrix | DFS con memo sulla cella | `@cache` su `(r, c)` |
| Distinct Subsequences | caratteri uguali: usa o salta | tabella con casi base `dp[i][0] = 1` |
| Edit Distance | tre operazioni → tre celle vicine | `1 + min(...)`, casi base `i` e `j` |
| Burst Balloons | DP su intervalli: quale scoppia per **ultimo** | padding `[1] + nums + [1]`, ciclo sulla lunghezza |
| Regular Expression Matching | `@cache` su `(i, j)`, caso `x*` | `j + 1 < len(p) and p[j + 1] == "*"` |

## Sintassi al volo

| Serve | Sintassi |
|---|---|
| tabella m × n | `dp = [[0] * n for _ in range(m)]` (**mai** `[[0] * n] * m`) |
| tabella con riga/colonna vuota | `dp = [[0] * (n + 1) for _ in range(m + 1)]` |
| memo su due parametri | `@cache` · `def dp(i: int, j: int) -> int` |
| ultima cella | `dp[-1][-1]` o `dp[m][n]` |
| riga precedente sola | `prev = [...]` · `cur = [0] * n` · `prev = cur` |
| intervalli per lunghezza crescente | `for length in range(2, n + 1): for l in range(n - length + 1): r = l + length - 1` |

## Creare una tabella 2D

`[[0] * n] * m` crea `m` riferimenti alla **stessa** riga: scrivere in una cella la cambia in tutte le righe.

In [1]:
wrong = [[0] * 3] * 2
wrong[0][0] = 9
right = [[0] * 3 for _ in range(2)]
right[0][0] = 9
print(wrong, right)  # -> [[9, 0, 0], [9, 0, 0]] [[9, 0, 0], [0, 0, 0]]

[[9, 0, 0], [9, 0, 0]] [[9, 0, 0], [0, 0, 0]]


## Griglie

Lo stato è la cella; arrivi da una cella "precedente" (sopra, sinistra). Prima riga e prima colonna sono casi base. Esempio: costo minimo di un percorso che va solo a destra o in basso.

In [2]:
cost = [
    [1, 3, 1],
    [1, 5, 1],
    [4, 2, 1],
]
rows, cols = len(cost), len(cost[0])
dp = [[0] * cols for _ in range(rows)]
for r in range(rows):
    for c in range(cols):
        if r == 0 and c == 0:
            dp[r][c] = cost[0][0]
            continue
        from_top = dp[r - 1][c] if r > 0 else float("inf")
        from_left = dp[r][c - 1] if c > 0 else float("inf")
        dp[r][c] = cost[r][c] + min(from_top, from_left)
print(dp[-1][-1])  # -> 7 (1 → 3 → 1 → 1 → 1)

7


**Una riga sola:** se `dp[r][c]` usa solo la riga sopra e la cella a sinistra, una lista `row` aggiornata da sinistra a destra basta: prima dell'aggiornamento `row[c]` è "sopra", `row[c - 1]` è già "a sinistra".

In [3]:
cols = 4
row = [1] * cols         # prima riga: un solo modo per ogni cella
for _ in range(2):       # altre 2 righe
    for c in range(1, cols):
        row[c] += row[c - 1]  # sopra + sinistra
print(row[-1])  # -> 10 percorsi in una griglia 3 × 4

10


## Due stringhe: tabella `(m + 1) × (n + 1)`

`dp[i][j]` riguarda i prefissi `a[:i]` e `b[:j]`. La riga e la colonna 0 rappresentano la stringa vuota (casi base). I caratteri "correnti" sono `a[i - 1]` e `b[j - 1]`.

Esempio: la **sottostringa** comune più lunga (contigua). Nota la differenza con la sottosequenza: qui, se i caratteri sono diversi, la catena si spezza (0).

In [4]:
a, b = "MUG-RED", "CAP-RED"
m, n = len(a), len(b)
dp = [[0] * (n + 1) for _ in range(m + 1)]
best = 0
for i in range(1, m + 1):
    for j in range(1, n + 1):
        if a[i - 1] == b[j - 1]:
            dp[i][j] = dp[i - 1][j - 1] + 1  # estende la diagonale
            best = max(best, dp[i][j])
        # else: 0 (già inizializzato)
print(best)  # -> 4 ("-RED")

4


| Cella vicina | Significato tipico |
|---|---|
| `dp[i - 1][j - 1]` | consumo un carattere da entrambe (match, sostituzione) |
| `dp[i - 1][j]` | consumo un carattere solo da `a` (cancellazione, "salto" in `a`) |
| `dp[i][j - 1]` | consumo un carattere solo da `b` (inserimento) |

## Top-down con `@cache`: spesso più semplice

Quando le transizioni sono tante o lo stato ha forme strane (somme negative, flag), scrivi la ricorsione su `(i, stato)` e aggiungi `@cache`. Esempio: macchina a stati per un magazzino che può tenere una sola unità alla volta.

In [5]:
from functools import cache

prices = [1, 2, 3, 0, 2]

@cache
def profit(day: int, holding: bool) -> int:
    if day >= len(prices):
        return 0
    wait = profit(day + 1, holding)
    if holding:
        act = prices[day] + profit(day + 1, False)  # vendi
    else:
        act = -prices[day] + profit(day + 1, True)  # compra
    return max(wait, act)

print(profit(0, False))  # -> 4: compra a 1, vendi a 3, compra a 0, vendi a 2

4


Per **contare i modi** con uno stato numerico che può essere negativo, un `dict` stato → conteggio aggiornato elemento per elemento evita la tabella con offset.

In [6]:
from collections import defaultdict

amounts = [1, 1, 1]
ways = {0: 1}  # somma corrente -> modi per ottenerla
for x in amounts:
    nxt: defaultdict[int, int] = defaultdict(int)
    for total, count in ways.items():
        nxt[total + x] += count
        nxt[total - x] += count
    ways = nxt
print(dict(sorted(ways.items())))  # -> {-3: 1, -1: 3, 1: 3, 3: 1}

{-3: 1, -1: 3, 1: 3, 3: 1}


## Combinazioni vs permutazioni: l'ordine dei cicli

Contare in quanti modi formare un importo:
- **monete fuori, importi dentro** → ogni moneta si considera una volta "in ordine": conti le **combinazioni**;
- **importi fuori, monete dentro** → conti le sequenze ordinate (**permutazioni**).

In [7]:
coins, amount = [1, 2], 3

combos = [1] + [0] * amount
for c in coins:
    for x in range(c, amount + 1):
        combos[x] += combos[x - c]

perms = [1] + [0] * amount
for x in range(1, amount + 1):
    for c in coins:
        if c <= x:
            perms[x] += perms[x - c]
print(combos[amount], perms[amount])  # -> 2 3  ({1,1,1}, {1,2} vs 1+1+1, 1+2, 2+1)

2 3


## Memo su griglia (DFS + cache)

Se un percorso può andare in tutte le direzioni ma i valori devono crescere strettamente, non ci sono cicli: la DFS dalla cella `(r, c)` ha un risultato fisso, e `@cache` lo calcola una volta sola per cella.

In [8]:
from functools import cache

heights = [[9, 9, 4], [6, 6, 8], [2, 1, 1]]
rows, cols = len(heights), len(heights[0])

@cache
def longest_from(r: int, c: int) -> int:
    best = 1
    for nr, nc in ((r + 1, c), (r - 1, c), (r, c + 1), (r, c - 1)):
        if 0 <= nr < rows and 0 <= nc < cols and heights[nr][nc] > heights[r][c]:
            best = max(best, 1 + longest_from(nr, nc))
    return best

print(max(longest_from(r, c) for r in range(rows) for c in range(cols)))  # -> 4 (1 → 2 → 6 → 9)

4


## DP su intervalli

`dp[l][r]` = risposta per il sotto-array `l..r`. Va riempita per **lunghezze crescenti**, perché un intervallo dipende da intervalli più corti. Dentro, provi ogni punto di divisione `k`. Un padding agli estremi (es. `[1] + nums + [1]`) evita i casi di bordo.

In [9]:
nums = [3, 1, 5]
vals = [1] + nums + [1]
n = len(vals)
print(vals)  # -> [1, 3, 1, 5, 1]

for length in range(2, n):            # distanza tra gli estremi esclusi
    for l in range(0, n - length):
        r = l + length
        middle = list(range(l + 1, r))  # candidati k strettamente dentro (l, r)
        if length == 2:
            print((l, r), middle)  # -> (0, 2) [1] / (1, 3) [2] / (2, 4) [3]

[1, 3, 1, 5, 1]
(0, 2) [1]
(1, 3) [2]
(2, 4) [3]


## Errori comuni

- Tabella 2D con `[[0] * n] * m`.
- Indici sfasati: con la tabella `(m + 1) × (n + 1)` il carattere è `a[i - 1]`, non `a[i]`.
- Casi base dimenticati: prima riga/colonna (griglie), `dp[i][0]` e `dp[0][j]` (stringhe).
- Coin Change II con i cicli invertiti: conta le permutazioni.
- DP su intervalli riempita per `l` crescente invece che per lunghezza: leggi celle non ancora calcolate.